### Widoki analityczne

- Przykład 1 – liczba odwołanych lotów per lotnisko z nazwą miasta

In [0]:
%sql
CREATE OR REPLACE VIEW flight_sales_gold.vw_airport_cancelled AS
SELECT
    f.date_key,
    ap.airport_name,
    ap.city,
    SUM(f.cancelled) AS num_cancelled
FROM flight_sales_silver.dim_route as r 
JOIN flight_sales_silver.dim_airports ap
    ON r.origin_airport_key = ap.airport_key
JOIN flight_sales_gold.fact_flights f
    ON r.route_key = f.route_key
GROUP BY f.date_key, ap.airport_name, ap.city
ORDER BY num_cancelled DESC;

In [0]:
%sql 
SELECT * FROM flight_sales_gold.vw_airport_cancelled
LIMIT 10; 


date_key,airport_name,city,num_cancelled
2015-02-23,Dallas/Fort Worth International Airport,Dallas-Fort Worth,642
2015-03-05,Dallas/Fort Worth International Airport,Dallas-Fort Worth,593
2015-02-28,Dallas/Fort Worth International Airport,Dallas-Fort Worth,570
2015-02-01,Chicago O'Hare International Airport,Chicago,540
2015-02-02,Chicago O'Hare International Airport,Chicago,534
2015-12-28,Chicago O'Hare International Airport,Chicago,514
2015-01-27,Gen. Edward Lawrence Logan International Airport,Boston,451
2015-01-27,LaGuardia Airport (Marine Air Terminal),New York,438
2015-02-09,Gen. Edward Lawrence Logan International Airport,Boston,431
2015-02-02,Gen. Edward Lawrence Logan International Airport,Boston,425


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

- Przykład 2 - liczba lotów per data z nazwą linii lotniczej 

In [0]:
%sql
CREATE OR REPLACE VIEW flight_sales_gold.vw_airline_daily_delay AS
SELECT
    f.date_key,
    a.airline_name,
    AVG(f.arrival_delay) AS avg_arrival_delay,
    AVG(f.departure_delay) AS avg_departure_delay,
    COUNT(*) AS num_flights
FROM flight_sales_gold.fact_flights f
JOIN flight_sales_silver.dim_airlines a
    ON f.airline_key = a.airline_key
GROUP BY f.date_key, a.airline_name;


In [0]:
%sql 
SELECT * FROM flight_sales_gold.vw_airline_daily_delay
LIMIT 10; 

date_key,airline_name,avg_arrival_delay,avg_departure_delay,num_flights
2015-07-14,Southwest Airlines Co.,3.172133526850508,7.963667820069205,3617
2015-07-11,Alaska Airlines Inc.,-2.9350912778904665,0.08519269776876268,493
2015-08-10,Southwest Airlines Co.,-4.438212634822804,2.50752688172043,3274
2015-07-04,Hawaiian Airlines Inc.,-9.923076923076923,-7.4125874125874125,143
2015-12-22,Southwest Airlines Co.,1.0618616866063613,6.266705573387803,3451
2015-12-26,Delta Air Lines Inc.,4.008870214752568,10.890289449112979,2154
2015-10-01,Virgin America,5.736196319018405,8.049079754601227,163
2015-12-31,Virgin America,-2.1797752808988764,8.314606741573034,178
2015-03-21,Spirit Air Lines,4.580912863070539,8.431535269709544,241
2015-08-01,Spirit Air Lines,25.254901960784313,28.212418300653596,306


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

- Przykład 3 - ile danego dnia odwołano lotów z powodu różnych przyczyn 

In [0]:
%sql
CREATE OR REPLACE VIEW flight_sales_gold.vw_cancellation_reason AS
SELECT 
    f.date_key,
    cr.code,
    cr.description,
    SUM(f.cancelled) AS num_cancelled
FROM flight_sales_gold.fact_flights f
JOIN flight_sales_silver.dim_cancellation_reason AS cr 
    ON f.cancellation_reason_key = cr.cancellation_reason_key
GROUP BY f.date_key, cr.code, cr.description;


In [0]:
%sql 
SELECT * FROM flight_sales_gold.vw_cancellation_reason
LIMIT 10;  

date_key,code,description,num_cancelled
2015-06-27,A,Carrier,126
2015-11-21,B,Weather,563
2015-04-19,A,Carrier,91
2015-02-14,C,National Air System,28
2015-04-09,B,Weather,316
2015-10-18,C,National Air System,5
2015-07-02,A,Carrier,115
2015-09-04,C,National Air System,31
2015-07-28,A,Carrier,110
2015-09-07,A,Carrier,13


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Dobowa granulacja 
- grupuje fakty po `date_key` i liczę miary 

In [0]:
%sql
-- Agregacja dobowa: średnie opóźnienia per dzień
SELECT
    date_key,
    AVG(arrival_delay) AS avg_arrival_delay,
    AVG(departure_delay) AS avg_departure_delay
FROM flight_sales_gold.fact_flights
GROUP BY date_key
ORDER BY date_key;


date_key,avg_arrival_delay,avg_departure_delay
2015-01-01,5.352495543672014,9.610896960711639
2015-01-02,9.838903743315509,12.649745269286754
2015-01-03,25.461860465116278,25.168418964491174
2015-01-04,31.975011015295525,31.567859384808536
2015-01-05,18.81131019036954,21.116838062197992
2015-01-06,21.299273998386663,22.486405036163944
2015-01-07,11.955428646448732,14.522800130847235
2015-01-08,13.316482498511215,16.40372752329702
2015-01-09,12.25561145510836,15.368481964894233
2015-01-10,1.9224748723018619,8.200148014143574
